# Classify your own bat recordings and spectrograms

[Open in Colab](https://colab.research.google.com/github/Kitware/batbot/blob/main/examples/notebooks/01_classify_your_files.ipynb) · [Open Codespaces](https://codespaces.new/Kitware/batbot) · [Developer guide](../../CONTRIBUTING.rst)

Bring **WAV files or existing JPG/PNG spectrograms**. Run top-to-bottom to obtain per-file predictions, retained processing images, top-class charts, file-level species counts, and downloadable JSON/CSV/PNG results. With no files supplied, an integrity-checked public example makes **Run All** work.

This is inference, not training. Scores are not calibrated probabilities, and file counts are **not numbers of calls, animals, or verified species detections**. Have domain experts review consequential identifications.

## 1. Environment: Codespaces, Colab, or local Jupyter

- **Codespaces:** choose this repository/your fork and the desired branch, let the dev-container setup finish, then select **Python (BatBot)**.
- **Colab:** open this notebook using **File → Open notebook → GitHub** or its Colab link. CPU works immediately; no GPU allocation is required.
- **Local:** install `python -m pip install -e ".[cpu,notebook]"` and select that environment's kernel.

The next cell installs into the active Python kernel when needed. It uses an existing checkout without switching branches or discarding edits; Colab clones a fresh checkout if needed. Before this PR is merged, set `SOURCE_REF` to the branch containing these files. Review code before running it.

Keep `RUNTIME_PROFILE="cpu"` for the quickest first run. For optional Colab GPU use, select a GPU runtime first, then choose `"auto"` (driver-aware), `"cuda12"` (ONNX Runtime 1.26/CUDA 12), or `"cuda13"` (1.31/CUDA 13). GPU dependencies are large, a compatible driver/device is required, and unavailable explicit providers fail clearly. Start a **fresh runtime** before changing CPU/GPU profiles; do not install both ONNX distributions. Native macOS can select `"coreml"`.

In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import subprocess
import sys

SOURCE_REF = "main"  # Before merge, use the branch containing these notebooks.
RUNTIME_PROFILE = "cpu"  # cpu, auto, cuda12, cuda13, or coreml (native macOS)
REPO_URL = "https://github.com/Kitware/batbot.git"

override = os.environ.get("BATBOT_NOTEBOOK_REPO")
REPO = Path(override).resolve() if override else next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "pyproject.toml").is_file() and (p / "batbot").is_dir()), None
)
if REPO is None:
    REPO = Path.cwd() / "batbot-tutorial-source"
    if not REPO.exists():
        clone_env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
        subprocess.check_call(
            ["git", "clone", "--depth", "1", "--no-checkout", REPO_URL, str(REPO)],
            env=clone_env,
        )
        # Fetch explicitly so SOURCE_REF can be a branch, tag, or exact commit SHA.
        subprocess.check_call(
            ["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", SOURCE_REF],
            env=clone_env,
        )
        subprocess.check_call(
            ["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"], env=clone_env
        )
helper = REPO / "examples/notebooks/notebook_utils.py"
if not helper.is_file():
    raise RuntimeError("This checkout lacks the tutorial helpers. Select the notebook's branch/tag.")
spec = importlib.util.spec_from_file_location("batbot_notebook_utils", helper)
nb = importlib.util.module_from_spec(spec)
spec.loader.exec_module(nb)
PROFILE, PROVIDER = nb.prepare_environment(REPO, RUNTIME_PROFILE)
print(f"Kernel: {sys.executable} (Python {sys.version.split()[0]})")
print(f"Using checkout: {REPO}. SOURCE_REF applies only when cloning a new checkout.")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from batbot import classifier

get_ipython().run_line_magic("matplotlib", "inline")

## 2. Bring your own files

**Codespaces/local Jupyter:** drag files into `data/input/` using the file explorer, or change `DATA_DIR` to an existing file/folder. Subfolders and mixed supported image/audio inputs are discovered recursively. Native paths, spaces, Unicode, and uppercase extensions are supported.

**Colab upload:** set `UPLOAD_FILES=True` and rerun the configuration/upload cells. Multiple files are accepted. **Drive:** set `MOUNT_DRIVE=True`, set `DATA_DIR=Path("/content/drive/MyDrive/your-folder")`, and authorize only if permitted. Uploads/Drive mounting are opt-in; neither happens on a default Run All.

Start with a small subset. For labeled evaluation preserve `LABEL/filename.wav` folders via a folder path/Drive; browser multi-file upload does not preserve ground-truth directories. Cloud session storage is temporary—download results before ending the session.

In [ ]:
import tempfile
from datetime import datetime, timezone

DATA_DIR = nb.native_path(os.environ.get("BATBOT_NOTEBOOK_INPUT_DIR", REPO / "data/input"))
OUTPUT_ROOT = nb.native_path(os.environ.get("BATBOT_NOTEBOOK_OUTPUT_DIR", REPO / "artifacts/notebooks"))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR = Path(tempfile.mkdtemp(prefix="run-", dir=OUTPUT_ROOT))
UPLOAD_FILES = False  # Colab: set True to open a multi-file picker.
MOUNT_DRIVE = False   # Colab: set True, then point DATA_DIR at your permitted Drive folder.
USE_EXAMPLE_IF_EMPTY = True
print("Input:", DATA_DIR)
print("Outputs:", RUN_DIR)

MAX_FILES = 8
BATCH_SIZE = 10          # Model windows per inference call.
INFERENCE_WORKERS = 1    # One reusable ONNX session; increase only after a baseline.
PREPROCESS_WORKERS = 1   # Safe notebook default; spawned worker functions live in BatBot modules.

In [ ]:
if MOUNT_DRIVE or UPLOAD_FILES:
    try:
        from google.colab import drive, files
    except ImportError as error:
        raise RuntimeError("Outside Colab, upload using the file explorer/JupyterLab sidebar.") from error
    if MOUNT_DRIVE:
        drive.mount("/content/drive")
    if UPLOAD_FILES:
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        files.upload(target_dir=str(DATA_DIR))

In [ ]:
inputs = nb.choose_inputs(
    DATA_DIR, REPO, OUTPUT_ROOT / "cache", maximum=MAX_FILES,
    allow_example=USE_EXAMPLE_IF_EMPTY,
)
display(pd.DataFrame({
    "file": [str(path) for path in inputs],
    "MiB": [round(path.stat().st_size / 2**20, 2) for path in inputs],
    "kind": [path.suffix.lower() for path in inputs],
}))

## 3. Run the real BatBot pipeline

For WAV inputs BatBot creates spectrograms with `fast_mode=False` and classifies the **full uncompressed JPEG chunks**. It also produces compressed chirp images/metadata for inspection; those are not the default model inputs. Existing image inputs bypass WAV preprocessing.

Images become overlapping **224 × 224 × 3 BGR uint8** windows. BatBot averages model scores over windows and combines multi-image WAV scores weighted by window count. One prediction is returned per input file. The detailed companion notebook exposes the STFT, gain, candidates, contours, masks, compression, and window scores.

In [ ]:
runner = classifier.Classifier(
    providers=[PROVIDER], batch_size=BATCH_SIZE, num_workers=INFERENCE_WORKERS,
)
report = runner.classify_bulk(
    inputs, input_type="auto", preprocess_workers=PREPROCESS_WORKERS,
    spectrogram_output=RUN_DIR / "spectrograms", progress=True,
)
table, scores = nb.result_tables(report)
display(pd.Series(report["summary"], name="summary"))
display(table)
if table["error"].notna().any():
    display(table.loc[table["error"].notna(), ["file", "error"]])
print("Active providers:", runner.session.get_providers())

## 4. Inspect processing images and class scores

The gallery shows the **actual model input images**, not screenshots of charts. WAV outputs are retained in this run's folder; original uploads are untouched. The next chart summarizes successful files only and includes `NOISE`. Failed inputs remain in the table and JSON report. A low score is not automatically noise or an unknown species—there is no automatic abstention threshold here.

In [ ]:
model_images = [
    Path(path)
    for result in report["results"] if "label" in result
    for path in result.get("spectrogram_paths", [result["path"]])
]
display(nb.show_images(model_images, RUN_DIR / "model_inputs.png", limit=4,
                       title="Actual spectrogram inputs used by the classifier"))
plt.close("all")
display(nb.plot_results(report, RUN_DIR / "classification_summary.png"))
plt.close("all")

In [ ]:
successful = [result for result in report["results"] if "label" in result]
if successful:
    selected = successful[0]  # Change this index to inspect another result.
    ranked = pd.Series(selected["scores"]).nlargest(8).sort_values()
    figure, axis = plt.subplots(figsize=(8, 4), layout="constrained")
    ranked.plot.barh(ax=axis, color="#087f8c")
    axis.set(xlim=(0, 1), xlabel="Model score", title=f"Top classes: {Path(selected['path']).name}")
    figure.savefig(RUN_DIR / "top_classes.png", dpi=140)
    display(figure)
    plt.close(figure)

## 5. Optional: compare against reviewed folder labels

Set the flag below only when each immediate parent folder is a valid model species code (for example `EPFU/recording.wav` or `NOISE/call.jpg`). Folder names must be **independently reviewed ground truth**, not copied from model predictions. The demo has no ground-truth label.

Accuracy below excludes failed files and prints their count; it is a small exploratory check, not a defensible performance estimate. Use independent held-out sites/nights/recordings and the labeled performance script for a full study.

In [ ]:
LABELED_FOLDERS = False
if LABELED_FOLDERS and successful:
    truth = [Path(result["path"]).parent.name for result in successful]
    unknown = sorted(set(truth) - set(classifier.CLASSES))
    if unknown:
        raise ValueError(f"Unknown folder labels: {unknown}. Use model species codes, or disable this step.")
    predicted = [result["label"] for result in successful]
    labels = [label for label in classifier.CLASSES if label in set(truth + predicted)]
    confusion = pd.crosstab(pd.Series(truth, name="Reviewed"), pd.Series(predicted, name="Predicted"))
    confusion = confusion.reindex(index=labels, columns=labels, fill_value=0)
    figure, axis = plt.subplots(figsize=(7, 6), layout="constrained")
    artist = axis.imshow(confusion, cmap="Blues")
    axis.set_xticks(range(len(labels)), labels, rotation=60)
    axis.set_yticks(range(len(labels)), labels)
    axis.set(xlabel="Predicted", ylabel="Reviewed folder label", title="Exploratory file-level confusion")
    figure.colorbar(artist, ax=axis, label="Files")
    figure.savefig(RUN_DIR / "confusion.png", dpi=140)
    display(figure)
    plt.close(figure)
    print("Accuracy on successful files:", np.mean(np.array(truth) == np.array(predicted)))
    print("Failed files excluded:", report["summary"]["failed"])

## 6. Export and reproduce

Exports include every class score and failures, plus model SHA-256, versions, providers, and inference settings. Keep a baseline before changing one parameter at a time. The archive contains this run's generated images/reports, **not the original input recordings**. Review it for sensitive filenames/derived data before sharing.

In Codespaces/JupyterLab, use the file explorer to download the ZIP. In Colab, set `DOWNLOAD_RESULTS=True` below for a browser download. These CSV/JSON exports are **not NABat upload templates**.

In [ ]:
provenance = nb.export_report(report, runner, RUN_DIR)
provenance["preprocess_workers"] = PREPROCESS_WORKERS
(RUN_DIR / "experiment_settings.json").write_text(
    json.dumps({"utc": datetime.now(timezone.utc).isoformat(),
                             "runtime_profile": PROFILE, **provenance}, indent=2), encoding="utf8"
)
archive = nb.archive_outputs(RUN_DIR)
display(pd.Series(provenance, name="provenance"))
print("Results archive:", archive)
DOWNLOAD_RESULTS = False
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(str(archive))

## Collaborate responsibly

Code contributions and reproducible prototypes are welcome in [BatBot](https://github.com/Kitware/batbot) and [Kitware BatAI](https://github.com/Kitware/batai). **Monitoring data contributions belong in the [USGS NABat program](https://www.usgs.gov/centers/fort-collins-science-center/science/north-american-bat-monitoring-program-nabat) database via the [Partner Portal](https://sciencebase.usgs.gov/nabat/), not GitHub.** Follow [NABat's current upload guidance](https://www.nabatmonitoring.org/upload-data).

These exports are exploratory results, not NABat-ready submissions, verified observations, or evidence of program approval. Share only permitted, non-sensitive information. Reports/debug images can contain paths and recording details. Clear notebook outputs before a pull request; never commit private audio, locations, credentials, or generated archives.